# Predictive Maintenance — Exploratory Data Analysis
**Dataset:** AI4I 2020 Predictive Maintenance Dataset  
**Author:** Devansh Gahlot  
**Goal:** Understand the data, find patterns related to machine failure, and prepare for modelling.

---

## 0. Setup — install and import libraries

In [ ]:
# Run this once to install dependencies
# !pip install pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from IPython.display import display

# Clean plot style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

print('Libraries loaded.')

## 1. Load the data

**How to get the data:**  
1. Go to https://www.kaggle.com/datasets/stephanmatzka/predictive-maintenance-dataset-ai4i-2020  
2. Download `ai4i2020.csv`  
3. Place it in a `data/` folder next to this notebook  

Alternatively, download directly from UCI:

In [ ]:
# Option A: load from local file (after downloading)
# df = pd.read_csv('../data/ai4i2020.csv')

# Option B: load directly from UCI (no download needed)
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/00601/ai4i2020.csv'
df = pd.read_csv(url)

print(f'Dataset loaded: {df.shape[0]:,} rows, {df.shape[1]} columns')
df.head()

## 2. First look — shape, types, missing values

In [ ]:
print('=== Column names ===')
print(df.columns.tolist())

print('\n=== Data types ===')
print(df.dtypes)

print('\n=== Missing values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values — clean dataset!')

In [ ]:
# Clean up column names — remove spaces and brackets for easier coding
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('[', '', regex=False)
    .str.replace(']', '', regex=False)
)

print('Cleaned column names:')
print(df.columns.tolist())

In [ ]:
# Summary statistics for numerical columns
df.describe().round(2)

## 3. Target variable — how many failures are there?

This is a key question for any classification problem. We need to know if the classes are balanced.

In [ ]:
# Identify the target column (machine failure)
# The column is likely named 'machine_failure' after our rename
target_col = 'machine_failure'

counts = df[target_col].value_counts()
pct = df[target_col].value_counts(normalize=True) * 100

summary = pd.DataFrame({'count': counts, 'percentage': pct.round(1)})
summary.index = ['No failure (0)', 'Failure (1)']
print(summary)

# Plot
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(['No Failure', 'Failure'], counts.values, color=['#4DA8A0', '#E07B5A'], width=0.5)
ax.set_title('Class distribution — machine failure', fontweight='bold')
ax.set_ylabel('Count')
for bar, count, pct_val in zip(bars, counts.values, pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
            f'{count:,}\n({pct_val:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, counts.max() * 1.2)
sns.despine()
plt.tight_layout()
plt.savefig('class_distribution.png', dpi=120)
plt.show()

print(f'\nClass imbalance ratio: 1:{counts[0]//counts[1]}')
print('NOTE: This imbalance means we MUST use F1/Recall for evaluation, not accuracy.')

## 4. Feature distributions — what do the sensor readings look like?

In [ ]:
# Identify numeric sensor features
# Adjust these names based on what printed in step 2
sensor_features = [
    'air_temperature_k',
    'process_temperature_k',
    'rotational_speed_rpm',
    'torque_nm',
    'tool_wear_min'
]

# Filter to only columns that actually exist
sensor_features = [c for c in sensor_features if c in df.columns]
print('Using features:', sensor_features)

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(sensor_features):
    ax = axes[i]
    # Overlay histograms: failure vs no-failure
    for label, color, name in [(0, '#4DA8A0', 'No failure'), (1, '#E07B5A', 'Failure')]:
        subset = df[df[target_col] == label][feat]
        ax.hist(subset, bins=40, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(feat.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.legend(fontsize=8)
    sns.despine(ax=ax)

# Hide unused subplot
for j in range(len(sensor_features), len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Feature distributions: failure vs no-failure', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Observation: Look for features where the orange (failure) distribution clearly shifts away from blue.')

## 5. Correlation heatmap — which features are related to each other?

In [ ]:
corr_cols = sensor_features + [target_col]
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr, dtype=bool))  # Hide upper triangle
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
    center=0, vmin=-1, vmax=1, square=True,
    linewidths=0.5, ax=ax, annot_kws={'size': 10}
)
ax.set_title('Correlation matrix — sensor features + failure', fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120)
plt.show()

# Print correlation with target specifically
print('\nCorrelation of each feature with machine_failure:')
print(corr[target_col].drop(target_col).sort_values(ascending=False).round(3))

## 6. Failure modes — what types of failure exist?

In [ ]:
# The dataset has specific failure type columns — find them
failure_cols = [c for c in df.columns if 'failure' in c and c != target_col]
print('Failure type columns found:', failure_cols)

if failure_cols:
    failure_counts = df[failure_cols].sum().sort_values(ascending=False)
    
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.barh(failure_counts.index, failure_counts.values, color='#E07B5A', height=0.5)
    ax.set_xlabel('Number of occurrences')
    ax.set_title('Failure modes — frequency', fontweight='bold')
    for bar, val in zip(bars, failure_counts.values):
        ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
                str(val), va='center', fontsize=10)
    sns.despine()
    plt.tight_layout()
    plt.savefig('failure_modes.png', dpi=120)
    plt.show()

## 7. Boxplots — sensor readings at failure vs no-failure

In [ ]:
fig, axes = plt.subplots(1, len(sensor_features), figsize=(16, 5))

palette = {0: '#4DA8A0', 1: '#E07B5A'}

for ax, feat in zip(axes, sensor_features):
    sns.boxplot(
        data=df, x=target_col, y=feat,
        palette=palette, width=0.5, ax=ax,
        flierprops=dict(marker='o', markersize=2, alpha=0.3)
    )
    ax.set_xticklabels(['No failure', 'Failure'])
    ax.set_title(feat.replace('_', ' ').replace('k', '(K)').replace('rpm', '(rpm)').title(), fontsize=10)
    ax.set_xlabel('')
    sns.despine(ax=ax)

fig.suptitle('Sensor readings at failure vs no-failure', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('boxplots_failure.png', dpi=120)
plt.show()

## 8. Machine type — does failure rate vary by machine type?

In [ ]:
type_col = 'type'
if type_col in df.columns:
    type_failure = df.groupby(type_col)[target_col].agg(['sum', 'count'])
    type_failure['failure_rate_%'] = (type_failure['sum'] / type_failure['count'] * 100).round(1)
    type_failure.columns = ['failures', 'total', 'failure_rate_%']
    print('Failure rate by machine type:')
    print(type_failure)

    fig, ax = plt.subplots(figsize=(5, 4))
    ax.bar(type_failure.index, type_failure['failure_rate_%'], color=['#4DA8A0', '#7ABFD6', '#E07B5A'], width=0.4)
    ax.set_ylabel('Failure rate (%)')
    ax.set_title('Failure rate by machine type', fontweight='bold')
    ax.yaxis.set_major_formatter(mtick.PercentFormatter())
    sns.despine()
    plt.tight_layout()
    plt.savefig('failure_by_type.png', dpi=120)
    plt.show()

## 9. EDA summary — key findings

Write your observations here after running the notebook. This will go into your README and LinkedIn write-up.

In [ ]:
print('=== EDA SUMMARY ===')
print(f'Dataset size: {df.shape[0]:,} rows x {df.shape[1]} columns')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Failure rate: {df[target_col].mean()*100:.1f}%')
print(f'Class imbalance: {counts[0]:,} normal vs {counts[1]:,} failures')
print()
print('Features most correlated with failure:')
print(corr[target_col].drop(target_col).abs().sort_values(ascending=False).round(3))
print()
print('Next step: Feature engineering + model training (notebook 02)')

---
## ✅ What's next?

Once you've completed this EDA:

1. **Write down 3–5 bullet points of key findings** in plain English — these become your LinkedIn post
2. **Commit this notebook to GitHub** with `git add`, `git commit`, `git push`
3. **Move to notebook 02** — feature engineering (temperature difference, power = torque × speed, wear ratio)
4. **Then notebook 03** — train Logistic Regression baseline, then Random Forest

**Questions to answer in your write-up:**
- Which sensor feature is most predictive of failure?
- Is there a threshold value of tool wear where failure risk jumps?
- Do high torque + low RPM combinations (= high stress) correlate with failure?
- Which machine type fails most often — and why might that be from an engineering perspective?